[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indianspeedster/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B,N,D = Q.shape
    output = torch.zeros_like(Q)

    for q_start in range(0, N, block_size):
      q_end = min(N, q_start+block_size)
      q_block = Q[:,q_start:q_end,:]
      Bq = q_end - q_start
      m = torch.full((B,Bq,1), float("-inf"))
      l = torch.zeros((B,Bq,1))
      acc = torch.zeros((B, Bq, D))
      for k_start in range(0, N, block_size):
        k_end = min(k_start+block_size, N)
        k_block = K[:, k_start:k_end, :]
        v_block = V[:, k_start:k_end, :]
        scores = (q_block @ k_block.transpose(-2,-1))/D**0.5
        block_max = scores.max(dim=-1, keepdim=True).values
        m_new = torch.maximum(block_max, m)
        alpha = torch.exp(m-m_new)
        p = torch.exp(scores-m_new)
        l_new = alpha*l + p.sum(dim=-1, keepdim=True)
        acc = alpha*acc + p@v_block
        m = m_new
        l = l_new
      out_block = acc/l
      output[:, q_start:q_end, :] = out_block
    return output





In [4]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (34.5ms)
  ✅ [2/4] Non-aligned block size (3.4ms)
  ✅ [3/4] Block size invariant (3.4ms)
  ✅ [4/4] Gradient flow (43.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (84.5ms total)
  Progress saved. Run status() to see your dashboard.

